# 🕸️ Five Multi-Agent Architectures in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Network architecture** - every agent can hand off to every other agent, with no central coordinator
2. **Supervisor architecture** - one node makes all routing decisions and workers always return to it
3. **Supervisor with tool-calling** - agents exposed as *tools*, letting the model's own tool-calling do the routing
4. **Hierarchical architecture** - compiled team subgraphs nested under a top-level coordinator
5. **Custom workflow architecture** - explicit, mostly-fixed paths with a branch point in the middle
6. **`Command(goto=...)`** - the primitive underneath all five, which lets a node choose its own successor

## Prerequisites
- Core LangGraph concepts: `StateGraph`, `START`/`END`, nodes and edges — see `03_LangGraph_Fundamentals/`
- A `.env` at the repository root with credentials for whichever provider `get_llm()` resolves to
- Packages: `langgraph`, `langchain`, `langchain-openai`, `python-dotenv`

> **Read this one first.** It is the map of the territory: the only notebook in this track that
> builds all five architectures side by side. Every other multi-agent notebook here is a deep dive
> into one of them.

## 📐 The five architectures at a glance

| # | Architecture | Who decides what runs next | Typical use |
|---|---|---|---|
| 1 | **Network** | Each agent decides for itself | Open-ended collaboration, peer review |
| 2 | **Supervisor** | One central node | Clear division of labour, auditable routing |
| 3 | **Supervisor w/ tool-calling** | The model, via tool selection | Intent routing, support desks |
| 4 | **Hierarchical** | A coordinator over team coordinators | Org-shaped problems, many specialists |
| 5 | **Custom workflow** | Mostly the graph; one dynamic branch | Pipelines with a decision point |

**Origin:** LangGraph's multi-agent guides; conceptual ancestors are AutoGen (Microsoft, 2023) and CrewAI (2023).

### Key Concepts:
- **`Command(goto=..., update=...)`**: a node's return value that does two jobs at once — write to state *and* name the next node. This is what makes non-linear control flow possible without pre-wiring every edge.
- **`MessagesState`**: LangGraph's built-in state with a single `messages` channel whose reducer appends. Because it appends, nodes return only their *new* messages — never the whole list.
- **Static edges vs. dynamic routing**: `add_edge` fixes a path at build time; `Command(goto=)` picks one at run time. Most real systems use both.

---
## 🔧 Part 0: Setup

One setup cell for the whole notebook. The original version re-imported the same LangGraph symbols
in every section; they are consolidated here so each architecture cell contains only its own logic.

`show_graph()` is defined here too — all five architectures are visualised the same way, and the
Mermaid renderer calls a remote service, so it falls back to ASCII when that is unavailable.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, Credentials, LLM, and Graph Rendering
# ============================================================================

# --- Standard library ---
from typing import Annotated, Literal

# --- Third-party ---
from dotenv import load_dotenv
from IPython.display import Image, display

# --- LangChain ---
from langchain.agents import create_agent
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_core.tools import tool

# --- LangGraph ---
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

# --- Project helpers ---
from helpers.utils import get_llm

load_dotenv()

# --- Safety cap -----------------------------------------------------------
# LangGraph counts SUPER-STEPS, not "iterations", and the default cap is
# 10,007 - high enough that a graph which never reaches END will churn for a
# very long time before failing. Every invoke below passes this limit
# explicitly so a routing bug surfaces in seconds instead of minutes.
# (AgentExecutor's `max_iterations` has no equivalent here; this is it.)
RECURSION_LIMIT = 25

# Platform-aware factory. Swap the provider by uncommenting an alternative.
llm = get_llm()
# llm = get_llm(provider="openai", model="gpt-4o")
# llm = get_llm(provider="groq")


def show_graph(compiled_graph):
    """Render a compiled graph, falling back to ASCII if the Mermaid service is unreachable."""
    graph = compiled_graph.get_graph()
    try:
        display(Image(graph.draw_mermaid_png(draw_method=MermaidDrawMethod.API)))
    except Exception as exc:
        print(f"⚠️ Mermaid render unavailable ({type(exc).__name__}); showing ASCII instead\n")
        print(graph.draw_ascii())


print(f"✅ Setup complete — LLM, show_graph(), recursion cap = {RECURSION_LIMIT} super-steps")

---
## 🕸️ Part 1: Network Architecture

Every agent may hand off to **any** other agent. There is no coordinator: each node inspects state
and names its own successor. That makes the topology many-to-many and the control flow entirely
emergent.

The three agents below — researcher, writer, reviewer — each return a `Command` whose `goto` is
computed from what is already in `messages`.

> **Note**: The `Literal[...]` in each return annotation is not decoration. It declares the set of
> nodes this one may jump to, and LangGraph uses it to draw the edges you will see in the diagram.

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Agent Definitions
# ============================================================================
# Each agent reads shared state, produces its own contribution, and names the
# next agent to run. Nobody coordinates them.


def researcher_agent(state: MessagesState) -> Command[Literal["writer_agent", "reviewer_agent", END]]:
    """Research a topic, then decide who should act on the findings."""
    messages = state.get("messages", [])
    research_result = "Research complete: Found 5 key points about AI architectures"

    # Hand off to the writer unless research has already been recorded.
    if not messages or "research complete" not in str(messages[-1]).lower():
        next_agent = "writer_agent"
    else:
        next_agent = END

    return Command(goto=next_agent, update={"messages": [f"[RESEARCHER]: {research_result}"]})


def writer_agent(state: MessagesState) -> Command[Literal["researcher_agent", "reviewer_agent", END]]:
    """Draft content from the research, then route to review."""
    messages = state.get("messages", [])
    content = "Draft article written based on research findings"

    if any("research complete" in str(msg).lower() for msg in messages):
        next_agent = "reviewer_agent"
    else:
        next_agent = "researcher_agent"  # nothing to write from yet

    return Command(goto=next_agent, update={"messages": [f"[WRITER]: {content}"]})


def reviewer_agent(state: MessagesState) -> Command[Literal["researcher_agent", "writer_agent", END]]:
    """Review the draft and either approve it or send the work back."""
    messages = state.get("messages", [])
    review = "Article reviewed and approved for publication"

    if any("draft article" in str(msg).lower() for msg in messages):
        next_agent = END
    else:
        next_agent = "researcher_agent"

    return Command(goto=next_agent, update={"messages": [f"[REVIEWER]: {review}"]})


print("✅ Network agents defined: researcher, writer, reviewer")

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Build and Compile
# ============================================================================
# Only ONE static edge exists — START to the researcher. Every other transition
# is chosen at runtime by a Command(goto=...).

builder = StateGraph(MessagesState)
builder.add_node("researcher_agent", researcher_agent)
builder.add_node("writer_agent", writer_agent)
builder.add_node("reviewer_agent", reviewer_agent)

builder.add_edge(START, "researcher_agent")
network = builder.compile()

print("✅ Network compiled — 3 nodes, 1 static edge, the rest dynamic")

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Visualize
# ============================================================================
# The dashed edges are the ones LangGraph inferred from the Literal annotations.

show_graph(network)

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Run
# ============================================================================

result = network.invoke(
    {"messages": ["Start research project"]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Message trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

Each agent appended exactly one message and named its own successor, so the path
`researcher → writer → reviewer → END` was never wired anywhere — it emerged from three independent
decisions.

**The trade-off:** flexibility with no termination guarantee. Nothing in a network stops two agents
bouncing work between themselves forever; only the agents' own conditions end the run. Compare that
against the supervisor in Part 2, where exactly one node decides when work is finished.

#### 🛡️ Seeing the safety cap work

The paragraph above is not hypothetical, so here is the failure and the guard against it.

Two agents below hand work to each other unconditionally — a routing bug you could plausibly write
by accident. Without a cap this runs **10,007 super-steps** before LangGraph gives up. With
`recursion_limit` it stops in a fraction of a second and tells you what happened.

> **Note**: this caps *steps*, not wall-clock time. Twenty-five steps of a graph that calls an LLM
> at every node is still slow — it is a runaway-loop guard, not a timeout.

In [ ]:
# ============================================================================
# SAFETY CAP: A Deliberately Non-Terminating Network
# ============================================================================
# ping and pong route to each other forever and never reach END.


def ping(state: MessagesState) -> Command[Literal["pong"]]:
    """Always hands off to pong. Never ends."""
    return Command(goto="pong", update={"messages": ["[PING]"]})


def pong(state: MessagesState) -> Command[Literal["ping"]]:
    """Always hands off to ping. Never ends."""
    return Command(goto="ping", update={"messages": ["[PONG]"]})


loop_builder = StateGraph(MessagesState)
loop_builder.add_node("ping", ping)
loop_builder.add_node("pong", pong)
loop_builder.add_edge(START, "ping")
runaway = loop_builder.compile()

try:
    runaway.invoke({"messages": []}, config={"recursion_limit": RECURSION_LIMIT})
    print("❌ Unexpected: the runaway graph terminated on its own")
except GraphRecursionError:
    print(f"✅ Caught GraphRecursionError after {RECURSION_LIMIT} super-steps")
    print("   Without the cap this would have run 10,007 steps before failing.")
    print("   Catch it explicitly when a partial result is still useful; otherwise")
    print("   let it raise, because a graph that cannot stop is a bug, not a slow run.")

---
## 🧑‍💼 Part 2: Supervisor Architecture

A single `supervisor` node makes every routing decision, and workers always return to it. Control
flow becomes a star: supervisor → worker → supervisor → worker → … → `END`.

This is the most common production shape, because the routing logic lives in exactly one place you
can read, log and test.

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Supervisor and Worker Nodes
# ============================================================================
# Note the asymmetry in the return types: the supervisor may go to any worker
# or END, but each worker may only go back to the supervisor.


def supervisor(state: MessagesState) -> Command[Literal["content_creator", "editor", END]]:
    """Decide which worker runs next, or end the run."""
    messages = state.get("messages", [])

    if not messages:
        return Command(goto="content_creator")

    last_message = str(messages[-1]).lower()

    if "content created" in last_message and "edited" not in last_message:
        return Command(goto="editor")
    elif "edited and polished" in last_message:
        return Command(goto=END)
    else:
        return Command(goto="content_creator")


def content_creator(state: MessagesState) -> Command[Literal["supervisor"]]:
    """Produce the first draft, then report back."""
    content = "Content created: 'Introduction to Multi-Agent Systems'"
    return Command(goto="supervisor", update={"messages": [f"[CREATOR]: {content}"]})


def editor(state: MessagesState) -> Command[Literal["supervisor"]]:
    """Polish the draft, then report back."""
    edit = "Content edited and polished for publication"
    return Command(goto="supervisor", update={"messages": [f"[EDITOR]: {edit}"]})


print("✅ Supervisor and workers defined: supervisor, content_creator, editor")

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Build and Compile
# ============================================================================

builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor)
builder.add_node("content_creator", content_creator)
builder.add_node("editor", editor)

builder.add_edge(START, "supervisor")
supervisor_system = builder.compile()

print("✅ Supervisor system compiled")

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Visualize
# ============================================================================

show_graph(supervisor_system)

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Run
# ============================================================================
# Starting from an EMPTY message list, so the supervisor takes its "no messages
# yet" branch and bootstraps the run itself.

result = supervisor_system.invoke(
    {"messages": []},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Message trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

Only the workers' output appears in the trace — the supervisor routes without writing to state.
That is deliberate and worth copying: a router that does not add messages keeps the transcript
readable.

**The trade-off:** every decision funnels through one node, which is both the strength (one place to
audit) and the weakness (a single point of failure, and a bottleneck when workers could have run in
parallel).

---
## 🛠️ Part 3: Supervisor with Tool-Calling

Instead of writing routing logic by hand, expose each agent as a **tool** and let the model's own
tool-calling choose. The supervisor becomes a standard agent; "routing" is just tool selection.

`InjectedState` is the piece that makes these real agents rather than plain functions: the graph's
state is injected into the tool call without the model having to produce it as an argument.

> **API note:** this section originally used `create_react_agent` from `langgraph.prebuilt`, which
> is deprecated since LangGraph v1.0 and scheduled for removal in v2.0. It now uses `create_agent`
> from `langchain.agents`. Two things changed together — the import moved, *and* the `prompt`
> argument was renamed `system_prompt` — so renaming only the function raises `TypeError`.

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Agents Exposed as Tools
# ============================================================================
# The docstring is not documentation here — it is the tool description the model
# reads to decide which specialist to call. Write it for the model.


@tool
def billing_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle billing-related queries and issues."""
    return f"Billing Agent: Processed query '{query}' - Issue resolved with account credit"


@tool
def technical_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle technical support and troubleshooting."""
    return f"Technical Agent: Analyzed '{query}' - Solution provided with step-by-step guide"


@tool
def general_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle general inquiries and information requests."""
    return f"General Agent: Addressed '{query}' - Information provided with helpful resources"


agent_tools = [billing_agent, technical_agent, general_agent]

print("✅ Agent-tools defined:", [t.name for t in agent_tools])

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Build
# ============================================================================
# No StateGraph and no routing code — create_agent supplies the loop, and the
# model's tool selection IS the routing decision.

tool_supervisor = create_agent(llm, tools=agent_tools)

print("✅ Tool-calling supervisor ready")

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Run
# ============================================================================
# A deliberately mixed query — billing AND technical — to see whether the model
# routes to more than one specialist.

result = tool_supervisor.invoke(
    {"messages": [("user", "I'm having trouble with my bill and the app keeps crashing")]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🤖 Conversation trace:")
for message in result["messages"]:
    kind = type(message).__name__
    content = getattr(message, "content", str(message))
    calls = [tc["name"] for tc in getattr(message, "tool_calls", []) or []]
    print(f"   {kind}: {content or '(no text)'}")
    if calls:
        print(f"      ↳ tool calls: {calls}")

### What to notice

Check which specialists actually got called. A mixed query *may* produce two tool calls, or the
model may decide one covers it — that variability is the point of this architecture. Routing quality
is now a prompt-and-description problem, not a code problem.

**The trade-off:** you gain flexibility and lose determinism. In Part 2 you could read the routing
rules; here you can only influence them through tool descriptions, and the same input can route
differently between runs.

---
## 🏢 Part 4: Hierarchical Architecture

Two levels of supervision. Each **team** is its own `StateGraph`, compiled independently, then added
to the parent graph as a single node — a compiled graph is a valid node because it has the same
invoke interface as a function.

The structure below is an org chart: a top coordinator over a dev team and a QA team, each with
their own supervisor and their own specialists.

### Key Concepts:
- **Subgraph as a node**: `builder.add_node("dev_team", dev_team)` where `dev_team` is a *compiled graph*. The parent never sees inside it.
- **Shared state contract**: this works only because every level uses `MessagesState`. If teams used different schemas you would need adapter code at each boundary.
- **Test teams in isolation first**: each team compiles and runs on its own, which is how you debug a hierarchy without running the whole thing.

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Development Team Subgraph
# ============================================================================
# A complete, self-contained graph — its own supervisor and its own workers.


def dev_supervisor(state: MessagesState) -> Command[Literal["frontend_dev", "backend_dev", END]]:
    """Route work within the development team until both specialists are done."""
    messages = state.get("messages", [])
    all_messages = " ".join(str(msg).lower() for msg in messages)

    backend_done = "[backend]" in all_messages and "complete" in all_messages
    frontend_done = "[frontend]" in all_messages and "complete" in all_messages

    if not backend_done:
        return Command(goto="backend_dev")
    elif not frontend_done:
        return Command(goto="frontend_dev")
    else:
        return Command(goto=END)


def backend_dev(state: MessagesState) -> Command[Literal["dev_supervisor"]]:
    """Backend specialist."""
    return Command(goto="dev_supervisor", update={"messages": ["[BACKEND]: Development complete"]})


def frontend_dev(state: MessagesState) -> Command[Literal["dev_supervisor"]]:
    """Frontend specialist."""
    return Command(goto="dev_supervisor", update={"messages": ["[FRONTEND]: Development complete"]})


dev_builder = StateGraph(MessagesState)
dev_builder.add_node("dev_supervisor", dev_supervisor)
dev_builder.add_node("backend_dev", backend_dev)
dev_builder.add_node("frontend_dev", frontend_dev)
dev_builder.add_edge(START, "dev_supervisor")
dev_team = dev_builder.compile()

print("✅ Dev team compiled — it is a standalone graph, runnable on its own")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: QA Team Subgraph
# ============================================================================
# Same shape as the dev team. Building both the same way is what lets the parent
# treat them interchangeably.


def qa_supervisor(state: MessagesState) -> Command[Literal["tester", "security_analyst", END]]:
    """Route work within the QA team until testing and security are both done."""
    messages = state.get("messages", [])
    all_messages = " ".join(str(msg).lower() for msg in messages)

    testing_done = "[tester]" in all_messages and "complete" in all_messages
    security_done = "[security]" in all_messages and "complete" in all_messages

    if not testing_done:
        return Command(goto="tester")
    elif not security_done:
        return Command(goto="security_analyst")
    else:
        return Command(goto=END)


def tester(state: MessagesState) -> Command[Literal["qa_supervisor"]]:
    """Functional testing specialist."""
    return Command(goto="qa_supervisor", update={"messages": ["[TESTER]: Testing complete"]})


def security_analyst(state: MessagesState) -> Command[Literal["qa_supervisor"]]:
    """Security review specialist."""
    return Command(
        goto="qa_supervisor", update={"messages": ["[SECURITY]: Security analysis complete"]}
    )


qa_builder = StateGraph(MessagesState)
qa_builder.add_node("qa_supervisor", qa_supervisor)
qa_builder.add_node("tester", tester)
qa_builder.add_node("security_analyst", security_analyst)
qa_builder.add_edge(START, "qa_supervisor")
qa_team = qa_builder.compile()

print("✅ QA team compiled")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Test Each Team in Isolation
# ============================================================================
# Do this BEFORE assembling the hierarchy. A team that misbehaves here will be
# far harder to diagnose once it is one node inside a larger graph.

for name, team in [("dev_team", dev_team), ("qa_team", qa_team)]:
    outcome = team.invoke(
        {"messages": [f"Kick off {name}"]},
        config={"recursion_limit": RECURSION_LIMIT},
    )
    print(f"🔧 {name}: {len(outcome['messages'])} messages")
    for message in outcome["messages"]:
        print(f"      {message}")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Top-Level Coordinator and Assembly
# ============================================================================
# The compiled teams are added as ordinary nodes. Static edges return control to
# the top supervisor after each team finishes.


def top_supervisor(state: MessagesState) -> Command[Literal["dev_team", "qa_team", END]]:
    """Coordinate the two teams: development first, then QA."""
    messages = state.get("messages", [])

    if not messages:
        return Command(goto="dev_team")

    all_messages = " ".join(str(msg).lower() for msg in messages)

    dev_complete = (
        "[backend]" in all_messages
        and "[frontend]" in all_messages
        and all_messages.count("development complete") >= 2
    )
    qa_complete = (
        "[tester]" in all_messages
        and "[security]" in all_messages
        and "testing complete" in all_messages
        and "security analysis complete" in all_messages
    )

    if not dev_complete:
        return Command(goto="dev_team")
    elif not qa_complete:
        return Command(goto="qa_team")
    else:
        return Command(goto=END)


builder = StateGraph(MessagesState)
builder.add_node("top_supervisor", top_supervisor)
builder.add_node("dev_team", dev_team)  # a COMPILED GRAPH used as a node
builder.add_node("qa_team", qa_team)

builder.add_edge(START, "top_supervisor")
builder.add_edge("dev_team", "top_supervisor")
builder.add_edge("qa_team", "top_supervisor")

hierarchical_system = builder.compile()

print("✅ Hierarchy compiled — 2 levels, 3 supervisors, 4 specialists")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Visualize
# ============================================================================
# The parent graph shows three nodes. The teams' internals are deliberately
# hidden — that encapsulation is the whole benefit of subgraphs.

show_graph(hierarchical_system)

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Run
# ============================================================================
result = hierarchical_system.invoke(
    {"messages": ["Start new project"]},
    # A hierarchy takes many more super-steps than a flat graph: each team run
    # is several steps of its own, so the flat cap is doubled here.
    config={"recursion_limit": RECURSION_LIMIT * 2},
)

print("🔧 Full trace across both levels:")
for i, message in enumerate(result["messages"], 1):
    print(f"   {i}. {message}")

### What to notice

Messages from both teams land in one flat transcript, because parent and subgraphs share the same
`MessagesState`. That shared contract is what lets a compiled graph drop in as a node with no
adapter code — and it is also why the top supervisor can inspect team output directly.

**The trade-off:** encapsulation costs you visibility. The parent cannot see *why* a team decided
anything, only what it appended. Raising `recursion_limit` is the tell: hierarchies take many more
steps, and step budget becomes something you have to think about.

---
## 🔀 Part 5: Custom Workflow Architecture

Mostly a fixed pipeline, with one genuine decision point. `document_parser → content_analyzer →
(text_processor | image_processor) → summarizer`.

This is the architecture most real systems actually need: the sequence is known, and only one or two
places require a judgement call.

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Pipeline Stages
# ============================================================================
# Every stage but one has a single possible successor. content_analyzer is the
# only real branch — note it is the only Literal with two options.


def document_parser(state: MessagesState) -> Command[Literal["content_analyzer"]]:
    """Stage 1 — parse the incoming document."""
    return Command(
        goto="content_analyzer",
        update={"messages": ["[PARSER]: Document parsed - detected format: PDF, 10 pages"]},
    )


def content_analyzer(state: MessagesState) -> Command[Literal["text_processor", "image_processor"]]:
    """Stage 2 — the branch point: inspect content and route accordingly."""
    content_type = "mixed"  # in a real system this would be detected, not hardcoded

    if content_type in ["text", "mixed"]:
        return Command(
            goto="text_processor",
            update={"messages": ["[ANALYZER]: Content analyzed - contains text and images"]},
        )
    return Command(
        goto="image_processor",
        update={"messages": ["[ANALYZER]: Content analyzed - images only"]},
    )


def text_processor(state: MessagesState) -> Command[Literal["summarizer"]]:
    """Stage 3a — extract and clean text."""
    return Command(
        goto="summarizer",
        update={"messages": ["[TEXT_PROC]: Text extracted and cleaned - 2,500 words"]},
    )


def image_processor(state: MessagesState) -> Command[Literal["summarizer"]]:
    """Stage 3b — analyse figures and diagrams."""
    return Command(
        goto="summarizer",
        update={"messages": ["[IMAGE_PROC]: Images processed - 5 charts and 3 diagrams analyzed"]},
    )


def summarizer(state: MessagesState) -> Command[Literal[END]]:
    """Stage 4 — converge both branches into one summary."""
    return Command(
        goto=END,
        update={"messages": ["[SUMMARIZER]: Final summary generated - key insights extracted"]},
    )


print("✅ Workflow stages defined: parser → analyzer → (text | image) → summarizer")

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Build and Compile
# ============================================================================

builder = StateGraph(MessagesState)
builder.add_node("document_parser", document_parser)
builder.add_node("content_analyzer", content_analyzer)
builder.add_node("text_processor", text_processor)
builder.add_node("image_processor", image_processor)
builder.add_node("summarizer", summarizer)

builder.add_edge(START, "document_parser")
custom_workflow = builder.compile()

print("✅ Custom workflow compiled — 5 stages, 1 branch point")

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Visualize
# ============================================================================
# This is the only diagram of the five that looks like a flowchart, because it
# is the only architecture that mostly IS one.

show_graph(custom_workflow)

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Run
# ============================================================================

result = custom_workflow.invoke(
    {"messages": ["Process uploaded document"]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Pipeline trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

Only one branch ran — `image_processor` never executed, because `content_analyzer` sent "mixed"
content down the text path. Both branches converge on `summarizer`, so the graph has a diamond shape
rather than a fork that never rejoins.

**The trade-off:** predictability at the cost of adaptability. This graph does the same thing every
time apart from one decision, which is exactly what you want for a document pipeline and exactly
what you do not want for open-ended research.

---
## 📝 Summary

### 1. One primitive underneath all five
- **`Command(goto=..., update=...)`** writes to state and names the next node in a single return. Every architecture here is a different discipline imposed on that one capability.
- **`Literal[...]` return annotations** declare a node's legal successors, and LangGraph reads them to build the graph you see rendered.

### 2. The five architectures, compared

| Architecture | Control | Fails when | Reach for it when |
|---|---|---|---|
| **Network** | Distributed | Nothing guarantees termination | Agents genuinely need to negotiate |
| **Supervisor** | Centralised | The supervisor is a bottleneck | Routing must be auditable |
| **Tool-calling supervisor** | Model-driven | Routing is non-deterministic | Intent classification is the hard part |
| **Hierarchical** | Layered | Step budgets grow; internals are opaque | The problem is org-shaped |
| **Custom workflow** | Graph-driven | Too rigid for open-ended work | The sequence is known in advance |

### 3. Design lessons worth carrying forward
- **A router that writes nothing keeps transcripts readable** — the Part 2 supervisor never appends a message.
- **Test subgraphs in isolation before assembling** — Part 4 runs each team alone first, which is the only practical way to debug a hierarchy.
- **One shared state schema removes adapter code** — the hierarchy composes only because every level speaks `MessagesState`.
- **Always pass `recursion_limit`** — the default is 10,007 super-steps, so an unguarded routing bug burns minutes before it fails. `AgentExecutor`'s `max_iterations` has no LangGraph equivalent; this is it.
- **Raise `recursion_limit` deliberately, not reflexively** — needing it is a signal about how many steps your design really takes.

### Next Steps
- `Production_Course_Multi_Agent/01_multi_agent.ipynb` — the supervisor pattern built slowly, with a Pydantic `RouteDecision` replacing the string matching used here.
- `Production_Course_Multi_Agent/06_hierarchical_agents.ipynb` — hierarchy done properly, with a purpose-built `TeamState` instead of `MessagesState`.
- `02_Multi_Agent_Swarm/01_Multi_Agent_Swarm.ipynb` — the network idea taken to production, with real handoff tools and persistent memory.

> **A note on the demos:** every agent here returns a hardcoded string, and routing is decided by
> substring matching on the transcript. That is deliberate — it keeps all five topologies visible
> without LLM latency or cost in the way. Only Part 3 calls a model. Treat these as diagrams you can
> execute, not as production code.